# Notebook 00 — Setup do SQL Server

Cria o banco **DespesaPublicaDB** e carrega as **11 tabelas** a partir dos CSVs da pasta `data/`.

**Sem ODBC.** Toda comunicação com o SQL Server é feita via:
- `sqlcmd` dentro do container Docker (para DDL — CREATE DATABASE, CREATE TABLE)
- **JDBC** pelo PySpark (para DML — INSERT via `df.write.jdbc`)

O driver JDBC é baixado automaticamente pelo Spark via Maven — **nenhuma instalação extra no SO**.

```
data/  *.csv  ──►  PySpark JDBC  ──►  SQL Server (DespesaPublicaDB)
```

In [ ]:
import os
import subprocess
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

SERVER   = os.getenv('SQL_SERVER',   'localhost')
PORT     = os.getenv('SQL_PORT',     '1433')
DATABASE = os.getenv('SQL_DATABASE', 'DespesaPublicaDB')
USER     = os.getenv('SQL_USER',     'sa')
PASSWORD = os.getenv('SQL_PASSWORD', 'SqlServer@2024!')

# JDBC URLs
JDBC_MASTER = (
    f'jdbc:sqlserver://{SERVER}:{PORT};'
    'databaseName=master;encrypt=true;trustServerCertificate=true'
)
JDBC_DB = (
    f'jdbc:sqlserver://{SERVER}:{PORT};'
    f'databaseName={DATABASE};encrypt=true;trustServerCertificate=true'
)
JDBC_PROPS = {
    'user': USER,
    'password': PASSWORD,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

DATA_DIR = '../data'

print(f'SQL Server : {SERVER}:{PORT}')
print(f'Banco      : {DATABASE}')
print('Sem ODBC   : comunicacao 100% via JDBC + sqlcmd no container')


SQL Server : localhost:1433
Banco      : DespesaPublicaDB
Sem ODBC   : comunicacao 100% via JDBC + sqlcmd no container


## 1. SparkSession com driver JDBC SQL Server

In [ ]:
# O pacote mssql-jdbc é baixado automaticamente do Maven Central
# Não precisa instalar nenhum driver no sistema operacional
spark = (
    SparkSession.builder
    .appName('DespesaPublica-Setup')
    .config(
        'spark.jars.packages',
        'com.microsoft.sqlserver:mssql-jdbc:12.4.2.jre11'
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} — driver JDBC SQL Server carregado via Maven')


26/05/05 13:29:27 WARN Utils: Your hostname, DESKTOP-6KNCQNJ resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 13:29:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/mendax/spark-delta-minio-despesa/spark-delta-minio-despesa/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mendax/.ivy2/cache
The jars for the packages stored in: /home/mendax/.ivy2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e4738a31-de56-4b62-8e0a-41420f198eff;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre11 in central
downloading https://repo1.maven.org/maven2/com/microsoft/sqlserver/mssql-jdbc/12.4.2.jre11/mssql-jdbc-12.4.2.jre11.jar ...
	[SUCCESSFUL ] com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre11!mssql-jdbc.jar (252ms)
:: resolution report :: resolve 880ms :: artifacts dl 258ms
	:: modules in use:
	com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre11 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------------------------------------

Spark 3.5.3 — driver JDBC SQL Server carregado via Maven


## 2. Criar o banco DespesaPublicaDB

> Usamos `sqlcmd` **dentro do container Docker** para executar DDL. Isso evita qualquer dependência de ODBC ou biblioteca nativa no host.

In [ ]:
def sqlcmd(query: str, db: str = 'master') -> str:
    """Executa SQL via sqlcmd dentro do container sqlserver-despesa."""
    cmd = [
        'docker', 'exec', 'sqlserver-despesa',
        '/opt/mssql-tools18/bin/sqlcmd',
        '-S', f'localhost,1433',
        '-U', USER, '-P', PASSWORD,
        '-d', db, '-C', '-Q', query
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print(sqlcmd(
    "IF NOT EXISTS (SELECT name FROM sys.databases WHERE name='DespesaPublicaDB') "
    "BEGIN CREATE DATABASE DespesaPublicaDB PRINT 'Banco criado!' END "
    "ELSE PRINT 'Banco ja existe.'"
))


Banco criado!


## 3. Criar as 11 tabelas

In [ ]:
DDL = [
  ('orgao', """
    IF OBJECT_ID('orgao','U') IS NULL
    CREATE TABLE orgao (
      id_orgao      INT PRIMARY KEY,
      codigo_orgao  VARCHAR(10)  NOT NULL,
      nome_orgao    VARCHAR(200) NOT NULL,
      sigla         VARCHAR(20)  NOT NULL,
      ativo         BIT DEFAULT 1
    )"""),
  ('unidade', """
    IF OBJECT_ID('unidade','U') IS NULL
    CREATE TABLE unidade (
      id_unidade      INT PRIMARY KEY,
      id_orgao        INT NOT NULL REFERENCES orgao(id_orgao),
      codigo_unidade  VARCHAR(20)  NOT NULL,
      nome_unidade    VARCHAR(300) NOT NULL,
      sigla           VARCHAR(30)  NOT NULL
    )"""),
  ('programa', """
    IF OBJECT_ID('programa','U') IS NULL
    CREATE TABLE programa (
      id_programa      INT PRIMARY KEY,
      codigo_programa  VARCHAR(20)  NOT NULL,
      nome_programa    VARCHAR(200) NOT NULL,
      ano_inicio       INT         NOT NULL,
      ativo            BIT DEFAULT 1
    )"""),
  ('acao', """
    IF OBJECT_ID('acao','U') IS NULL
    CREATE TABLE acao (
      id_acao      INT PRIMARY KEY,
      id_programa  INT NOT NULL REFERENCES programa(id_programa),
      codigo_acao  VARCHAR(20)  NOT NULL,
      nome_acao    VARCHAR(300) NOT NULL
    )"""),
  ('fonte_recurso', """
    IF OBJECT_ID('fonte_recurso','U') IS NULL
    CREATE TABLE fonte_recurso (
      id_fonte      INT PRIMARY KEY,
      codigo_fonte  VARCHAR(10)  NOT NULL,
      descricao     VARCHAR(200) NOT NULL
    )"""),
  ('credor', """
    IF OBJECT_ID('credor','U') IS NULL
    CREATE TABLE credor (
      id_credor     INT PRIMARY KEY,
      cnpj          VARCHAR(20)  NOT NULL,
      razao_social  VARCHAR(200) NOT NULL,
      municipio     VARCHAR(100),
      uf            CHAR(2),
      ativo         BIT DEFAULT 1
    )"""),
  ('empenho', """
    IF OBJECT_ID('empenho','U') IS NULL
    CREATE TABLE empenho (
      id_empenho           INT PRIMARY KEY,
      numero_empenho       VARCHAR(30)  NOT NULL,
      id_unidade           INT NOT NULL REFERENCES unidade(id_unidade),
      id_acao              INT NOT NULL REFERENCES acao(id_acao),
      id_fonte             INT NOT NULL REFERENCES fonte_recurso(id_fonte),
      id_credor            INT NOT NULL REFERENCES credor(id_credor),
      data_empenho         DATE        NOT NULL,
      valor_empenho        DECIMAL(18,2) NOT NULL,
      tipo_empenho         VARCHAR(30),
      modalidade_licitacao VARCHAR(50),
      situacao             VARCHAR(30),
      descricao            VARCHAR(500)
    )"""),
  ('item_empenho', """
    IF OBJECT_ID('item_empenho','U') IS NULL
    CREATE TABLE item_empenho (
      id_item         INT PRIMARY KEY,
      id_empenho      INT NOT NULL REFERENCES empenho(id_empenho),
      descricao_item  VARCHAR(200) NOT NULL,
      quantidade      INT          NOT NULL,
      valor_unitario  DECIMAL(18,2) NOT NULL,
      valor_total     DECIMAL(18,2) NOT NULL
    )"""),
  ('liquidacao', """
    IF OBJECT_ID('liquidacao','U') IS NULL
    CREATE TABLE liquidacao (
      id_liquidacao     INT PRIMARY KEY,
      id_empenho        INT NOT NULL REFERENCES empenho(id_empenho),
      numero_liquidacao VARCHAR(30)  NOT NULL,
      data_liquidacao   DATE        NOT NULL,
      valor_liquidado   DECIMAL(18,2) NOT NULL,
      situacao          VARCHAR(30)
    )"""),
  ('pagamento', """
    IF OBJECT_ID('pagamento','U') IS NULL
    CREATE TABLE pagamento (
      id_pagamento          INT PRIMARY KEY,
      id_liquidacao         INT NOT NULL REFERENCES liquidacao(id_liquidacao),
      id_credor             INT NOT NULL REFERENCES credor(id_credor),
      numero_ordem_bancaria VARCHAR(30) NOT NULL,
      data_pagamento        DATE        NOT NULL,
      valor_pago            DECIMAL(18,2) NOT NULL,
      banco                 VARCHAR(50),
      agencia               VARCHAR(10),
      situacao              VARCHAR(30)
    )"""),
  ('historico_preco', """
    IF OBJECT_ID('historico_preco','U') IS NULL
    CREATE TABLE historico_preco (
      id_historico          INT PRIMARY KEY,
      descricao_item        VARCHAR(200) NOT NULL,
      ano_referencia        INT         NOT NULL,
      valor_medio           DECIMAL(18,2) NOT NULL,
      quantidade_pesquisada INT,
      fonte_pesquisa        VARCHAR(100)
    )""")
]

for nome, ddl in DDL:
    out = sqlcmd(ddl, db=DATABASE)
    print(f'  {nome:20s} OK  {out}')

print('\nTodas as tabelas criadas!')


  orgao                OK  
  unidade              OK  
  programa             OK  
  acao                 OK  
  fonte_recurso        OK  
  credor               OK  
  empenho              OK  
  item_empenho         OK  
  liquidacao           OK  
  pagamento            OK  
  historico_preco      OK  

Todas as tabelas criadas!


## 4. Carga dos CSVs → SQL Server via PySpark JDBC

In [ ]:
# Ordem que respeita as foreign keys
TABELAS = [
    'orgao', 'unidade', 'programa', 'acao', 'fonte_recurso',
    'credor', 'empenho', 'item_empenho', 'liquidacao', 'pagamento', 'historico_preco'
]

print('=== CARGA: CSV -> SQL Server (JDBC) ===\n')

for tabela in TABELAS:
    # Limpar antes de inserir (seguro para re-execucao)
    sqlcmd(f'DELETE FROM {tabela}', db=DATABASE)

    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv(f'{DATA_DIR}/{tabela}.csv')
    )

    df.write.jdbc(
        url=JDBC_DB,
        table=tabela,
        mode='append',
        properties=JDBC_PROPS
    )
    print(f'  {tabela:20s} {df.count():4d} registros')

print('\nCarga concluida!')


=== CARGA: CSV -> SQL Server (JDBC) ===



26/05/05 13:29:46 WARN TOKEN: ConnectionID:2 ClientConnectionId: e82a93be-90d9-4c29-81f6-62c540701772: SQLServerConnection.supportsTransactions: Discarding unexpected TDS_COLMETADATA (0x81)


  orgao                   5 registros


26/05/05 13:29:49 WARN TOKEN: ConnectionID:5 ClientConnectionId: 80b043ce-f097-4e30-8ef1-4fb5db292ec6: SQLServerConnection.supportsTransactions: Discarding unexpected TDS_COLMETADATA (0x81)


  unidade                11 registros


...

26/05/05 13:29:55 WARN TOKEN: ConnectionID:26 ClientConnectionId: 4b11a532-76b3-4f42-b15f-8ea49fdf61f3: SQLServerConnection.supportsTransactions: Discarding unexpected TDS_COLMETADATA (0x81)


  liquidacao             40 registros


26/05/05 13:29:56 WARN TOKEN: ConnectionID:29 ClientConnectionId: 0f14e37c-aeb2-41da-93d0-6a708aabe064: SQLServerConnection.supportsTransactions: Discarding unexpected TDS_COLMETADATA (0x81)


  pagamento              35 registros
  historico_preco        15 registros

Carga concluida!


26/05/05 13:29:56 WARN TOKEN: ConnectionID:32 ClientConnectionId: 4f0bc519-825a-4a7c-bf2a-366ee709fd5f: SQLServerConnection.supportsTransactions: Discarding unexpected TDS_COLMETADATA (0x81)


## 5. Validação

In [ ]:
print('=== VALIDACAO — leitura de volta via JDBC ===\n')
for tabela in TABELAS:
    n = spark.read.jdbc(url=JDBC_DB, table=tabela, properties=JDBC_PROPS).count()
    print(f'  {tabela:20s} {n:4d} registros')

spark.stop()
print('\nSetup concluido — DespesaPublicaDB pronto!')


=== VALIDACAO — leitura de volta via JDBC ===

  orgao                   5 registros
  unidade                11 registros
  programa                8 registros
  acao                   12 registros
  fonte_recurso           6 registros
  credor                 15 registros
  empenho                50 registros
  item_empenho          126 registros
  liquidacao             40 registros
  pagamento              35 registros
  historico_preco        15 registros

Setup concluido — DespesaPublicaDB pronto!
